In [ ]:
import json
import pandas as pd

# Load CARD JSON
with open("card.json", "r") as f:
    card_data = json.load(f)

# Prefixes to keep
prefixes = ["IMI", "KPC", "GES", "TEM", "SHV", "CTXM", "VEB", "SME"]

data = []

for entry_id, entry in card_data.items():
    # Only process actual model records
    if not isinstance(entry, dict):
        continue
    if "model_name" not in entry:
        continue

    model_name = entry.get("model_name", "")

    # Keep only genes with desired prefixes
    if not any(model_name.startswith(prefix) for prefix in prefixes):
        continue

    # Navigate actual CARD structure
    model_sequences = entry.get("model_sequences", {})
    sequence_block = model_sequences.get("sequence", {})

    for seq_id, seq_entry in sequence_block.items():
        protein_seq = (
            seq_entry.get("protein_sequence", {})
            .get("sequence", "")
        )
        protein_acc = (
            seq_entry.get("protein_sequence", {})
            .get("accession", "")
        )
        dna_acc = (
            seq_entry.get("dna_sequence", {})
            .get("accession", "")
        )
        aro_name = entry.get("ARO_name", "")
        aro_accession = entry.get("ARO_accession", "")
        card_short_name = entry.get("CARD_short_name", "")

        if protein_seq:
            data.append({
                "entry_id": entry_id,
                "model_name": model_name,
                "CARD_short_name": card_short_name,
                "ARO_name": aro_name,
                "ARO_accession": aro_accession,
                "sequence_id": seq_id,
                "protein_accession": protein_acc,
                "dna_accession": dna_acc,
                "protein_sequence": protein_seq
            })

# Make dataframe
df = pd.DataFrame(data)
df = df.rename(columns={
    "model_name": "Gene",
    "protein_sequence": "Sequence"
})
df["Family"] = df["Gene"].str.extract(r"^([A-Z]+)")
df["Carbapenemase"] = df["Family"].isin(["IMI", "KPC", "GES", "SME"]).astype(int)
df = df[["Family", "Gene", "Sequence", "Carbapenemase"]]
df.head()
df.to_csv("card_sequences.csv", index=False)

,Family,Gene,Sequence,Carbapenemase
0,SHV,SHV-52,MRYIRLCIISLLAALPLAVHASPQPLEQIKQSESQLSGRVGMIEMD...,0
1,TEM,TEM-126,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDQLGARVGYIE...,0
2,TEM,TEM-72,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDKLGARVGYIE...,0
3,TEM,TEM-59,MSIQHFRVALIPFFAAFCLPVFAHPETLVKVKDAEDKLGARVGYIE...,0
4,KPC,KPC-10,MSLYRRLVLLSCLSWPLAGFSATALTNLVAEPFAKLEQDFGGSIGV...,1
